In [1]:
"""
Scott-Knott ESD (Non-Parametric) — análisis completo
Genera los tests estadísticos y las dos figuras para el artículo.

Uso:
    python sk_esd_completo.py

Requiere:
    pip install pandas numpy scipy matplotlib
    El CSV debe estar en la misma carpeta: resultados_completos.csv

Salida:
    - fig1_rank_summary.png   (conteos de Rank 1 y mejor mediana)
    - fig2_heatmap_ranks.png  (heatmap de ranks por dataset y métrica)
    - sk_esd_ranks_f1.csv
    - sk_esd_ranks_recall.csv
    - sk_esd_medians_f1.csv
    - sk_esd_medians_recall.csv
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────

CSV_PATH    = 'resultados.csv'
SAMPLERS    = ['corsmote', 'smote', 'borderline', 'adasyn', 'Original']
LABELS      = ['CorSMOTE', 'SMOTE', 'Borderline', 'ADASYN', 'Original']
METRICS     = ['F1', 'Recall', 'Precision']
CLIFF_THRES = 0.147   # Romano et al. (2006)
ALPHA       = 0.05
COLORS      = ['#2563EB', '#6B7280', '#EA580C', '#7C3AED', '#16A34A']

# ─────────────────────────────────────────────
# SCOTT-KNOTT ESD (fiel al paquete R oficial)
# ─────────────────────────────────────────────

def cliff_delta(x, y):
    x, y = np.array(x), np.array(y)
    more = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return (more - less) / (len(x) * len(y))

def all_pairs_negligible(left_arrays, right_arrays):
    """
    Replica la función diff() de PartitionNonParametric en R:
    comprueba todos los pares (i,j) entre grupos izquierdo y derecho.
    Si algún par tiene |delta| >= 0.147 → no negligible → separar.
    """
    for a in left_arrays:
        for b in right_arrays:
            if not np.all(a == b):
                if abs(cliff_delta(a, b)) >= CLIFF_THRES:
                    return False
    return True

def kruskal_split(arrays):
    best_split, best_stat = None, -np.inf
    for i in range(1, len(arrays)):
        left  = np.concatenate(arrays[:i])
        right = np.concatenate(arrays[i:])
        try:
            stat, _ = stats.kruskal(left, right)
        except Exception:
            stat = 0
        if stat > best_stat:
            best_stat, best_split = stat, i
    return best_split, best_stat

def sk_esd(data_dict):
    """
    Non-Parametric Scott-Knott ESD.
    Validado contra klainfo/ScottKnottESD (rama development) en R.

    Parámetros
    ----------
    data_dict : dict {sampler_name: array_of_values}

    Devuelve
    --------
    dict {sampler_name: rank}   (rank empieza en 1, 1 = mejor)
    """
    order   = sorted(data_dict.keys(), key=lambda k: np.median(data_dict[k]), reverse=True)
    ordered = [(k, np.array(data_dict[k])) for k in order]
    ranks   = {k: None for k in data_dict}
    current = [1]

    def recurse(group):
        if len(group) == 1:
            ranks[group[0][0]] = current[0]
            current[0] += 1
            return

        arrays    = [g[1] for g in group]
        split_idx, _ = kruskal_split(arrays)

        left_data  = np.concatenate(arrays[:split_idx])
        right_data = np.concatenate(arrays[split_idx:])

        try:
            _, p = stats.kruskal(left_data, right_data)
        except Exception:
            p = 1.0

        negligible = all_pairs_negligible(arrays[:split_idx], arrays[split_idx:])

        if p >= ALPHA or negligible:
            for name, _ in group:
                ranks[name] = current[0]
            current[0] += 1
            return

        recurse(group[:split_idx])
        recurse(group[split_idx:])

    recurse(ordered)
    return ranks

# ─────────────────────────────────────────────
# PIVOT
# Original solo tiene ratio=0; los samplers tienen 3 ratios.
# Se agrupa por (sampler, seed, Model) tomando la mediana de ratios
# → 30 seeds × 8 modelos = 240 obs por dataset, comparación justa.
# ─────────────────────────────────────────────

def build_pivot(df_ds, metric):
    agg = df_ds.groupby(['sampler', 'seed', 'Model'])[metric].median().reset_index()
    pivot = agg.pivot_table(
        index=['seed', 'Model'],
        columns='sampler',
        values=metric
    ).dropna(subset=SAMPLERS)
    return pivot

# ─────────────────────────────────────────────
# CARGA
# ─────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
df = df[df['sampler'].isin(SAMPLERS)].copy()
datasets = sorted(df['dataset'].unique())

print(f"Datasets: {len(datasets)}")
print(f"Samplers: {SAMPLERS}\n")

# ─────────────────────────────────────────────
# ANÁLISIS POR DATASET
# ─────────────────────────────────────────────

all_ranks   = {}
all_medians = {}

for metric in METRICS:
    all_ranks[metric]   = {}
    all_medians[metric] = {}

    print(f"{'='*85}")
    print(f"Métrica: {metric}")
    print(f"{'='*85}")
    print(f"{'Dataset':<28} {'corsmote':>14} {'smote':>14} {'borderline':>14} {'adasyn':>14} {'Original':>14}")
    print(f"{'─'*88}")

    for dataset in datasets:
        df_ds     = df[df['dataset'] == dataset]
        pivot     = build_pivot(df_ds, metric)
        data_dict = {s: pivot[s].values for s in SAMPLERS}
        ranks     = sk_esd(data_dict)
        medians   = {s: np.median(data_dict[s]) for s in SAMPLERS}

        all_ranks[metric][dataset]   = ranks
        all_medians[metric][dataset] = medians

        def fmt(s): return f"{medians[s]:.3f} (R{ranks[s]})"
        print(f"{dataset:<28} {fmt('corsmote'):>14} {fmt('smote'):>14} {fmt('borderline'):>14} {fmt('adasyn'):>14} {fmt('Original'):>14}")

    print(f"{'─'*88}")

    r1 = {s: sum(1 for ds in datasets if all_ranks[metric][ds][s]==1) for s in SAMPLERS}
    bm = {s: sum(1 for ds in datasets if all_medians[metric][ds][s]==max(all_medians[metric][ds].values())) for s in SAMPLERS}

    print(f"\nVeces Rank 1 (de {len(datasets)}):")
    for s in SAMPLERS: print(f"  {s:<15}: {r1[s]:>2}/{len(datasets)}")
    print(f"\nMejor mediana (de {len(datasets)}):")
    for s in SAMPLERS: print(f"  {s:<15}: {bm[s]:>2}/{len(datasets)}")
    print()

# ─────────────────────────────────────────────
# EXPORTAR CSVs
# ─────────────────────────────────────────────

for metric in METRICS:
    slug = metric.lower()
    pd.DataFrame([{'dataset': ds, **{s: all_ranks[metric][ds][s] for s in SAMPLERS}} for ds in datasets])\
      .to_csv(f'sk_esd_ranks_{slug}.csv', index=False)
    pd.DataFrame([{'dataset': ds, **{s: round(all_medians[metric][ds][s],4) for s in SAMPLERS}} for ds in datasets])\
      .to_csv(f'sk_esd_medians_{slug}.csv', index=False)
    print(f"→ Guardado: sk_esd_ranks_{slug}.csv")
    print(f"→ Guardado: sk_esd_medians_{slug}.csv")

# ─────────────────────────────────────────────
# FIGURA 1 — Rank 1 y mejor mediana por sampler
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
x = np.arange(5)
w = 0.35

r1_f1 = [sum(1 for ds in datasets if all_ranks['F1'][ds][s]==1)    for s in SAMPLERS]
r1_rc = [sum(1 for ds in datasets if all_ranks['Recall'][ds][s]==1) for s in SAMPLERS]
r1_pr = [sum(1 for ds in datasets if all_ranks['Precision'][ds][s]==1) for s in SAMPLERS]
bm_f1 = [sum(1 for ds in datasets if all_medians['F1'][ds][s]==max(all_medians['F1'][ds].values()))     for s in SAMPLERS]
bm_rc = [sum(1 for ds in datasets if all_medians['Recall'][ds][s]==max(all_medians['Recall'][ds].values())) for s in SAMPLERS]
bm_pr = [sum(1 for ds in datasets if all_medians['Precision'][ds][s]==max(all_medians['Precision'][ds].values())) for s in SAMPLERS]

for ax, r1, bm, title in zip(axes, [r1_f1, r1_rc, r1_pr], [bm_f1, bm_rc, bm_pr], ['F1-score', 'Recall', 'Precision']):
    b1 = ax.bar(x - w/2, r1, w, color=COLORS, alpha=0.88, edgecolor='white')
    b2 = ax.bar(x + w/2, bm, w, color=COLORS, alpha=0.45, edgecolor='white', hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS, fontsize=10)
    ax.set_ylim(0, 22)
    ax.set_ylabel('Datasets (out of 22)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.axhline(22, color='gray', lw=0.5, ls='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, str(int(h)),
                    ha='center', va='bottom', fontsize=8)
    patch_solid = mpatches.Patch(facecolor='#888', label='Times Rank 1')
    patch_hatch = mpatches.Patch(facecolor='#888', alpha=0.45, hatch='//', label='Best median')
    ax.legend(handles=[patch_solid, patch_hatch], fontsize=9, loc='upper right')

fig.suptitle('Scott-Knott ESD — Rank 1 and best median counts per sampler\n',
             fontsize=11, y=1.01)
fig.tight_layout()
fig.savefig('fig1_rank_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n→ Guardado: fig1_rank_summary.png")



Datasets: 22
Samplers: ['corsmote', 'smote', 'borderline', 'adasyn', 'Original']

Métrica: F1
Dataset                            corsmote          smote     borderline         adasyn       Original
────────────────────────────────────────────────────────────────────────────────────────
abalone-20_vs_8-9-10             0.637 (R1)     0.644 (R1)     0.650 (R1)     0.636 (R1)     0.599 (R2)
bank_account_fraud               0.542 (R1)     0.541 (R1)     0.544 (R1)     0.543 (R1)     0.511 (R2)
cicids2017                       0.991 (R1)     0.991 (R1)     0.990 (R2)     0.988 (R3)     0.991 (R1)
credit                           0.849 (R2)     0.850 (R2)     0.887 (R1)     0.845 (R3)     0.863 (R2)
diabetes                         0.713 (R2)     0.713 (R2)     0.710 (R2)     0.707 (R2)     0.720 (R1)
ecoli_binary                     0.895 (R1)     0.895 (R1)     0.878 (R1)     0.892 (R1)     0.892 (R1)
flare-F                          0.602 (R1)     0.593 (R2)     0.588 (R2)     0.593 (R2) 